# 37. KV Cache Scheduling | KV Cache 调度
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `KV Cache`, `调度` | **目标人群：** 推理优化学习者

---

## 本节导读

当多个请求共享前缀并持续生成时，KV Cache 会同时面临复用和容量压力。调度器需要记录每个缓存的大小、命中次数和最近访问时间，再根据这些状态决定缓存的保留顺序和驱逐顺序。

本节沿着“请求访问 → 缓存状态 → 价值评分 → 优先级队列 → 容量驱逐”的顺序理解 KV Cache 调度。学习后，你应能根据容量、访问热度和复用次数解释一个前缀为何被保留或驱逐。


**关键词：** `KV Cache 调度`、`缓存复用`、`容量驱逐`

---


## 前置阅读

**导语：** 进入本节前，先能读出一个缓存块的容量、命中和最近访问状态，再观察这些状态如何影响保留与驱逐顺序。
- [22. vLLM PagedAttention | vLLM 分页注意力](./22_vLLM_PagedAttention.ipynb)
- [34. Prefix Cache Matching and Reuse | Prefix Cache 匹配与复用](./34_Prefix_Cache_Matching_and_Reuse.ipynb)
- [36. Decode Scheduling | 解码调度](./36_Decode_Scheduling.ipynb)

---


### Step 1: 为什么 KV Cache 需要调度

多个请求同时生成时，KV Cache 不仅要保存可复用状态，还要在容量有限时决定保留谁、驱逐谁。前缀缓存提供复用线索，分页管理提供可分配的 block，请求访问过程持续更新命中次数和最近访问时间。

输入是缓存访问事件、单条缓存的大小与访问状态，以及全局容量；输出是可比较的缓存价值、驱逐顺序和容量快照。

| 观察入口 | 发生了什么 | 形成的调度依据 |
|---|---|---|
| 内容复用 | 同一前缀再次被访问 | 命中次数和复用价值 |
| 空间占用 | 新请求需要更多 KV Cache block | 当前容量和驱逐压力 |
| 访问热度 | 命中次数和最近访问不断变化 | 当前缓存价值和优先级 |

![KV Cache 调度概念图：前置机制、缓存状态与容量决策](../docs/public/02_PyTorch_Algorithms/37_kv_cache_scheduling_map.svg)


### Step 2: 缓存状态与容量账本

先把每段可复用缓存看成一条状态记录，再把所有记录放进同一份容量账本。单条记录描述一个前缀的占用和访问历史，全局字段描述当前容量；这些字段共同决定下一步的保留与驱逐。

| 字段层级 | 记录什么 | 对调度的作用 |
|---|---|---|
| 单条缓存：标识 | prefix 对应的缓存记录 | 判断访问是否命中同一前缀 |
| 单条缓存：大小 | 这段缓存占用多少字节 | 计算新增缓存带来的容量压力 |
| 单条缓存：命中次数 | 该前缀已经复用多少次 | 估计继续保留的复用价值 |
| 单条缓存：最近访问 | 最近一次访问发生的时间 | 反映当前访问热度 |
| 全局账本：容量 | 总容量和当前已分配容量 | 判断何时需要触发驱逐 |

### Step 3: 评分、堆队列与过期记录

Step 2 的状态字段进入价值评估：复用次数体现未来命中机会，最近访问体现当前热度，缓存大小体现容量成本。一种可解释的保留策略可将三者合成为比较分数：`score = 复用奖励 + 最近访问奖励 - 容量惩罚`。分数变化后，优先级队列中的旧记录仍可能存在，因此弹出时要核对它是否仍然有效。


| 机制 | 作用 | 设计时关注 |
|---|---|---|
| 评分 | 把缓存价值变成可比较的数字 | 复用、时间和容量共同影响分数 |
| 堆队列 | 让低优先级缓存可以先被找到 | 刷新时追加当前记录，保留可追溯状态 |
| 过期检查（stale check） | 弹出时重新核对当前 entry | 让驱逐依据跟随最新状态 |
| 驱逐 | 容量不足时释放低价值缓存 | 超过容量时选择驱逐或降级路径 |

![KV Cache 调度的评分、堆队列与过期记录](../docs/public/02_PyTorch_Algorithms/37_kv_cache_score_heap.svg)

### Step 4: 实现评分、堆刷新与容量驱逐

把前面的状态账本和评分机制落到 `KVCacheSchedulerSim`。题目要求你完成评分、堆状态更新、过期记录处理和新条目登记四个决策；快照导出由骨架提供，用于观察当前缓存状态。

| TODO / 实现对象 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| TODO 1 / `_score` | 合成复用、热度和容量成本的保留分数 | 复用和新近访问提高分数，容量成本降低分数 | 评分方向单调性 |
| TODO 2 / `_refresh_queue` | 将最新评分快照加入堆 | 低分优先弹出，保留历史记录供 stale check | 堆刷新与历史项 |
| TODO 3 / `_evict_until_fit` | 跳过过期记录并驱逐有效低分项 | 不使用旧状态，容量必须回到可容纳范围 | stale record 与容量守恒 |
| TODO 4 / `touch` | 创建并登记新缓存条目 | 初始命中次数为 1，更新容量账本 | 新增、复用和输入契约 |




In [ ]:
import heapq
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


In [ ]:
@dataclass(order=True)
class CacheEntry:
    priority: float
    last_used: int
    prefix: str = field(compare=False)
    hits: int = field(default=0, compare=False)
    bytes: int = field(default=0, compare=False)


class KVCacheSchedulerSim:
    """用优先级堆模拟可复用前缀的容量调度。"""

    def __init__(self, capacity_bytes: int = 1024):
        """初始化容量账本、缓存状态和优先级队列。"""
        if capacity_bytes <= 0:
            raise ValueError('capacity_bytes must be positive')
        self.capacity_bytes = capacity_bytes
        self.current_bytes = 0
        self.time = 0
        self.entries: Dict[str, CacheEntry] = {}
        self.queue: List[Tuple[float, int, str]] = []
        self.log: List[str] = []

    def _score(self, hits: int, size: int, last_used: int) -> float:
        """根据复用次数、访问热度和容量成本计算保留分数。"""
        recency = 1.0 / (1.0 + max(self.time - last_used, 0))
        reuse_bonus = float(hits)
        size_penalty = size / max(self.capacity_bytes, 1)
        # ==========================================
        # TODO 1: 计算 cache entry 的保留优先级
        # 提示：复用次数越多、访问越新，分数越高；占用空间越大，惩罚越强。
        # score：当前 cache entry 的保留优先级，必须是 float
        # 可用变量：reuse_bonus、recency、size_penalty
        # ==========================================
        # score = ???
        return score

    def _refresh_queue(self, prefix: str):
        """把当前 entry 的最新评分追加到优先级堆。"""
        entry = self.entries[prefix]
        entry.priority = self._score(entry.hits, entry.bytes, entry.last_used)
        # ==========================================
        # TODO 2: 把最新优先级写入堆队列
        # 提示：这里要弹出低 priority 的缓存，因此不要对 priority 取负。
        # queue_item：堆中的状态快照，字段顺序必须是 (priority, last_used, prefix)
        # ==========================================
        # queue_item = ???
        heapq.heappush(self.queue, queue_item)

    def _evict_until_fit(self, needed: int):
        """在新增缓存前驱逐低价值 entry，直到容量可以容纳它。"""
        while self.current_bytes + needed > self.capacity_bytes and self.entries:
            while self.queue:
                priority, last_used, prefix = heapq.heappop(self.queue)
                entry = self.entries.get(prefix)
                if entry is None:
                    continue
                # ==========================================
                # TODO 3: 跳过堆中的过期记录
                # 提示：entry 的 priority 或 last_used 已变化时，旧堆项不再有效。
                # is_stale：当前堆项是否已经落后于 entry 的最新状态
                # ==========================================
                # is_stale = ???
                if is_stale:
                    continue
                break
            else:
                entry = min(self.entries.values(), key=lambda e: (e.priority, e.last_used))
                prefix = entry.prefix

            self.current_bytes -= entry.bytes
            self.entries.pop(prefix, None)
            self.log.append(f"evict:{prefix}")

    def touch(self, prefix: str, bytes_: int):
        """访问一个前缀，更新命中状态或创建新的缓存 entry。"""
        if not isinstance(prefix, str) or not prefix:
            raise ValueError('prefix must be a non-empty string')
        if bytes_ <= 0:
            raise ValueError('bytes_ must be positive')
        if bytes_ > self.capacity_bytes:
            raise ValueError("single cache entry exceeds capacity")

        self.time += 1
        if prefix in self.entries:
            entry = self.entries[prefix]
            entry.hits += 1
            entry.last_used = self.time
            self._refresh_queue(prefix)
            self.log.append(f"reuse:{prefix}")
            return

        self._evict_until_fit(bytes_)
        # ==========================================
        # TODO 4: 创建新的 cache entry
        # 提示：新 entry 的 hits 从 1 开始，last_used 使用当前 time。
        # entry：新建的 CacheEntry，保存前缀、字节数、访问时间和初始 priority=0.0
        # ==========================================
        # entry = ???
        self.entries[prefix] = entry
        self.current_bytes += bytes_
        self._refresh_queue(prefix)
        self.log.append(f"add:{prefix}")

    def schedule(self, requests: List[Tuple[str, int]]) -> List[str]:
        """按给定访问顺序处理前缀请求并返回事件日志。"""
        for prefix, bytes_ in requests:
            self.touch(prefix, bytes_)
        return list(self.log)

    def snapshot(self) -> List[Tuple[str, int, float, int]]:
        """按保留价值从高到低导出当前缓存状态。"""
        # 快照仅用于展示当前状态；排序规则由骨架固定，避免把格式处理当作机制 TODO。
        ordered_entries = sorted(self.entries.values(), key=lambda e: (-e.priority, e.last_used, e.prefix))
        return [(e.prefix, e.bytes, round(e.priority, 4), e.hits) for e in ordered_entries]


### 测试

运行下面的测试单元，确认缓存评分、驱逐和快照输出都符合预期。

In [ ]:
# 机制测试：按输入契约、评分排序、堆刷新和容量守恒分别定位问题。
def _expect_value_error(action, label):
    try:
        action()
    except ValueError:
        return
    raise AssertionError(f'{label} 应拒绝非法输入')


def test_kv_cache_input_contract():
    """验证容量、prefix 和 entry 大小的输入契约。"""
    _expect_value_error(lambda: KVCacheSchedulerSim(capacity_bytes=0), 'capacity_bytes=0')
    sim = KVCacheSchedulerSim(capacity_bytes=128)
    _expect_value_error(lambda: sim.touch('', 8), '空 prefix')
    _expect_value_error(lambda: sim.touch('bad-size', 0), '非正 bytes')
    _expect_value_error(lambda: sim.touch('too-large', 129), '超过容量的 entry')


def test_kv_cache_score_and_snapshot():
    """验证复用热度、容量成本与快照排序的单调性。"""
    sim = KVCacheSchedulerSim(capacity_bytes=128)
    sim.touch('a', 40)
    sim.touch('a', 40)
    hot_score = sim._score(hits=3, size=16, last_used=sim.time)
    cold_score = sim._score(hits=1, size=16, last_used=0)
    small_score = sim._score(hits=1, size=16, last_used=sim.time)
    large_score = sim._score(hits=1, size=64, last_used=sim.time)
    assert hot_score > cold_score
    assert small_score > large_score
    snapshot = sim.snapshot()
    assert all(len(item) == 4 for item in snapshot)
    priorities = [item[2] for item in snapshot]
    assert priorities == sorted(priorities, reverse=True)


def test_kv_cache_heap_refresh_and_stale_entries():
    """验证重复刷新后旧堆项不会破坏当前 entry 集合。"""
    sim = KVCacheSchedulerSim(capacity_bytes=80)
    sim.touch('hot', 32)
    sim.touch('hot', 32)
    assert len(sim.queue) >= 2
    sim.touch('cold', 64)
    assert sim.current_bytes <= sim.capacity_bytes
    assert len(sim.entries) == len(set(sim.entries))


def test_kv_cache_eviction_and_capacity_conservation():
    """验证复用、驱逐和容量账本在连续调度后的守恒。"""
    sim = KVCacheSchedulerSim(capacity_bytes=128)
    log = sim.schedule([('a', 40), ('b', 48), ('a', 40), ('c', 56), ('d', 48), ('a', 40)])
    assert len(log) >= 6
    assert any(item.startswith('reuse:a') for item in log)
    assert any(item.startswith('evict:') for item in log)
    assert sim.current_bytes <= sim.capacity_bytes
    assert sim.current_bytes == sum(item.bytes for item in sim.entries.values())


def test_kv_cache_scheduler():
    """汇总四组机制测试；失败时保留具体断言上下文。"""
    try:
        test_kv_cache_input_contract()
        test_kv_cache_score_and_snapshot()
        test_kv_cache_heap_refresh_and_stale_entries()
        test_kv_cache_eviction_and_capacity_conservation()
        print('✅ KVCacheSchedulerSim 机制测试通过：输入、评分、堆状态和容量守恒均通过。')
    except NotImplementedError as e:
        raise NotImplementedError('请先完成 TODO 代码！') from e
    except (NameError, AttributeError) as e:
        raise NotImplementedError('请先完成 TODO 代码或检查字段名！') from e


test_kv_cache_scheduler()


## 参考代码与解析

### 代码


In [ ]:
# TODO：下面是题目区的参考实现。

@dataclass(order=True)
class CacheEntry:
    priority: float
    last_used: int
    prefix: str = field(compare=False)
    hits: int = field(default=0, compare=False)
    bytes: int = field(default=0, compare=False)


class KVCacheSchedulerSim:
    """用优先级堆模拟可复用前缀的容量调度。"""

    def __init__(self, capacity_bytes: int = 1024):
        """初始化容量账本、缓存状态和优先级队列。"""
        if capacity_bytes <= 0:
            raise ValueError('capacity_bytes must be positive')
        self.capacity_bytes = capacity_bytes
        self.current_bytes = 0
        self.time = 0
        self.entries: Dict[str, CacheEntry] = {}
        self.queue: List[Tuple[float, int, str]] = []
        self.log: List[str] = []

    def _score(self, hits: int, size: int, last_used: int) -> float:
        """根据复用次数、访问热度和容量成本计算保留分数。"""
        recency = 1.0 / (1.0 + max(self.time - last_used, 0))
        reuse_bonus = float(hits)
        size_penalty = size / max(self.capacity_bytes, 1)
        # ==========================================
        # TODO 1: 计算 cache entry 的保留优先级
        # 提示: 复用次数越多越该保留，越新越该保留，越大越需要惩罚
        # 可用变量: reuse_bonus、recency、size_penalty；结果应为 float
        # ==========================================
        score = reuse_bonus + 0.5 * recency - 0.25 * size_penalty
        return score

    def _refresh_queue(self, prefix: str):
        """把当前 entry 的最新评分追加到优先级堆。"""
        entry = self.entries[prefix]
        entry.priority = self._score(entry.hits, entry.bytes, entry.last_used)
        # ==========================================
        # TODO 2: 把最新优先级写入堆队列
        # 提示: 这里要弹出低 priority 的缓存，因此不要对 priority 取负
        # queue_item 应包含 (priority, last_used, prefix) 三个字段
        # ==========================================
        queue_item = (entry.priority, entry.last_used, prefix)
        heapq.heappush(self.queue, queue_item)

    def _evict_until_fit(self, needed: int):
        """在新增缓存前驱逐低价值 entry，直到容量可以容纳它。"""
        while self.current_bytes + needed > self.capacity_bytes and self.entries:
            while self.queue:
                priority, last_used, prefix = heapq.heappop(self.queue)
                entry = self.entries.get(prefix)
                if entry is None:
                    continue
                # ==========================================
                # TODO 3: 跳过堆中的过期记录
                # 提示: entry 的 priority 或 last_used 已变化时，旧堆项不再有效
                # 可比较堆顶的 priority、last_used 与 entry 的当前字段
                # ==========================================
                is_stale = (priority, last_used) != (entry.priority, entry.last_used)
                if is_stale:
                    continue
                break
            else:
                entry = min(self.entries.values(), key=lambda e: (e.priority, e.last_used))
                prefix = entry.prefix

            self.current_bytes -= entry.bytes
            self.entries.pop(prefix, None)
            self.log.append(f"evict:{prefix}")

    def touch(self, prefix: str, bytes_: int):
        """访问一个前缀，更新命中状态或创建新的缓存 entry。"""
        if not isinstance(prefix, str) or not prefix:
            raise ValueError('prefix must be a non-empty string')
        if bytes_ <= 0:
            raise ValueError('bytes_ must be positive')
        if bytes_ > self.capacity_bytes:
            raise ValueError("single cache entry exceeds capacity")

        self.time += 1
        if prefix in self.entries:
            entry = self.entries[prefix]
            entry.hits += 1
            entry.last_used = self.time
            self._refresh_queue(prefix)
            self.log.append(f"reuse:{prefix}")
            return

        self._evict_until_fit(bytes_)
        # ==========================================
        # TODO 4: 创建新的 cache entry
        # 提示: 新 entry 的 hits 从 1 开始，last_used 使用当前 time
        # entry 需要保存 prefix、bytes_、当前 time，并先使用 0.0 作为初始 priority
        # ==========================================
        entry = CacheEntry(priority=0.0, last_used=self.time, prefix=prefix, hits=1, bytes=bytes_)
        self.entries[prefix] = entry
        self.current_bytes += bytes_
        self._refresh_queue(prefix)
        self.log.append(f"add:{prefix}")

    def schedule(self, requests: List[Tuple[str, int]]) -> List[str]:
        """按给定访问顺序处理前缀请求并返回事件日志。"""
        for prefix, bytes_ in requests:
            self.touch(prefix, bytes_)
        return list(self.log)

    def snapshot(self) -> List[Tuple[str, int, float, int]]:
        """按保留价值从高到低导出当前缓存状态。"""
        ordered_entries = sorted(self.entries.values(), key=lambda e: (-e.priority, e.last_used, e.prefix))
        return [(e.prefix, e.bytes, round(e.priority, 4), e.hits) for e in ordered_entries]


### 解析

本题依次实现缓存评分、堆刷新、过期记录处理、新条目登记和快照导出。答案代码保留与题目区相同的控制流，只补全每处机制决策。

**TODO 1：计算缓存保留优先级**
- 复用次数和最近访问时间提高保留价值，条目大小增加驱逐成本。
- `recency = 1 / (1 + time_gap)` 让长期未访问的前缀逐渐降权；测试检查评分方向的单调性。

**TODO 2：刷新优先级堆**
- 将 `(priority, last_used, prefix)` 压入堆，低优先级条目先成为驱逐候选。
- 同一 prefix 可能留下多个历史记录，因此刷新本身不负责物理删除。

**TODO 3：跳过过期堆项**
- 弹出记录后重新和 `entries` 中的当前 `priority`、`last_used` 比较。
- 不一致时按懒删除处理，避免旧记录误删已更新的缓存；测试覆盖 stale record 和容量不变量。

**TODO 4：创建新的缓存条目**
- 新条目从当前时间和一次初始访问开始登记，再交给统一刷新逻辑计算优先级。
- 单个条目超过总容量时应拒绝；测试同时检查新增、复用和非法输入。

**测试如何定位问题**
- 评分与快照测试检查复用热度、容量成本和当前状态排序是否一致。
- 堆刷新与驱逐测试检查旧记录不会误删新状态，并验证 `current_bytes` 始终等于当前条目的字节数之和。


### Step 5: 可选 GPU：观察代表性 KV block 的显存占用

#### 5.1 环境、输入与固定条件

本步把 Step 2 的容量账本连接到一次可复核的 CUDA 分配实验：固定 block 形状和 dtype，改变 block 数量，比较理论字节数与实际分配峰值。评分、堆队列和驱逐顺序仍由 CPU 题目区验证。

确认当前 Notebook 使用 GPU 内核后，将配置单元的开关改为 `True`。执行单元会检查 CUDA、dtype 和参数范围，并保存运行条件与结果 JSON；真实 backend 的扩展实验可转到 [69](./69_Prefix_Caching_Benchmark.ipynb) 和 [70](./70_Serving_Scheduler_Benchmark.ipynb)。

| 实验路径 | 使用资产 | 学习者操作 | 可以验证 / 不能直接推出 |
| --- | --- | --- | --- |
| CPU 机制验证 | 题目区 `KVCacheSchedulerSim` | 运行评分、堆队列、过期记录、驱逐顺序和容量账本测试 | 调度状态与容量守恒 |
| GPU 环境预检 | `tools/environment_preflight.py`、当前 Notebook runtime | 检查 CUDA、GPU、显存和 dtype 支持 | 当前 runtime 是否可运行 |
| GPU block probe | 配置单元、执行单元和 JSON 输出 | 固定 block 形状，单独改变 block 数量、token 数或 dtype | 理论 footprint 与 CUDA 分配峰值；`synthetic_gpu_block_probe` |
| 真实 backend 扩展 | vLLM / SGLang、69 和 70 | 在匹配的单 GPU backend 中测量 KV Cache、并发和请求延迟 | backend 行为与服务指标 |
| 结果登记 | 本节最后的 GPU 实验记录表 | 每组新配置新增一行，不覆盖已有结果 | 形成可比较的 block footprint 记录 |



#### 5.2 配置与执行

配置单元只负责选择实验参数；执行单元负责环境检查、分配代表性 block、同步 CUDA、生成 JSON。每次只改变一个主要变量，便于把显存变化归因到 block 数量、token 数或 dtype。

| 实验内容 | 固定或改变的条件 | 观察目的 |
| --- | --- | --- |
| 代表性 block | GPU_BLOCK_TOKENS、KV head 数、head dim | 对照理论字节数和实际分配 |
| 容量压力 | GPU_BLOCK_COUNT | 观察 block 数量与峰值显存的关系 |
| dtype | GPU_DTYPE | 比较 dtype 对 block footprint 的影响 |
| 证据范围 | synthetic GPU block probe | 不推导真实 backend 的命中率、驱逐策略或吞吐 |


In [ ]:
"""配置单元：默认只检查配置，不启动 GPU。"""
RUN_GPU_BLOCK_PROBE = False  # 改为 True 才会分配代表性 KV block。
GPU_RESULT_PATH = 'benchmarks/results/37_kv_cache_block_probe.json'  # 相对项目根目录。
GPU_BLOCK_COUNT = 16  # 只改变容量压力时修改；先从小规模开始。
GPU_BLOCK_TOKENS = 128  # 每个 block 代表的 token 数。
GPU_KV_HEADS = 8
GPU_HEAD_DIM = 64
GPU_DTYPE = 'float16'  # 可改为 bfloat16，但必须确认硬件原生支持。
GPU_REPEATS = 3


In [ ]:
"""执行单元：只测代表性 block 的分配，不启动模型或 backend。"""
if RUN_GPU_BLOCK_PROBE:
    import json
    import time
    from pathlib import Path
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('GPU block probe 需要 CUDA；请先切换到 GPU runtime。')
    if GPU_DTYPE not in {'float16', 'bfloat16'}:
        raise ValueError('GPU_DTYPE 只能是 float16 或 bfloat16。')
    if GPU_DTYPE == 'bfloat16' and not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError('当前 GPU 不支持原生 BF16，请改用 float16。')
    if any(value < 1 for value in (GPU_BLOCK_COUNT, GPU_BLOCK_TOKENS, GPU_KV_HEADS, GPU_HEAD_DIM, GPU_REPEATS)):
        raise ValueError('block 数量、token 数、KV head 数、head dim 和 repeats 必须为正数。')

    dtype = getattr(torch, GPU_DTYPE)
    block_shape = (2, GPU_BLOCK_TOKENS, GPU_KV_HEADS, GPU_HEAD_DIM)
    bytes_per_block = 1
    for dim in block_shape:
        bytes_per_block *= dim
    bytes_per_block *= torch.empty((), dtype=dtype).element_size()
    device = torch.device('cuda')
    runs = []
    for repeat_index in range(GPU_REPEATS):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        blocks = [torch.empty(block_shape, device=device, dtype=dtype) for _ in range(GPU_BLOCK_COUNT)]
        torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        runs.append({'repeat': repeat_index, 'allocation_ms': round(elapsed_ms, 3), 'peak_memory_mb': round(peak_mb, 2), 'block_count': GPU_BLOCK_COUNT})
        del blocks

    project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir()), Path.cwd())
    output_path = project_root / GPU_RESULT_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)
    report = {
        'task': 'kv_cache_block_footprint_probe',
        'evidence_level': 'synthetic_gpu_block_probe',
        'runtime': {'device': torch.cuda.get_device_name(0), 'gpu_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2), 'torch': torch.__version__, 'torch_cuda': torch.version.cuda, 'native_bf16_supported': torch.cuda.is_bf16_supported(including_emulation=False)},
        'config': {'block_shape': block_shape, 'bytes_per_block_theoretical': bytes_per_block, 'total_bytes_theoretical': bytes_per_block * GPU_BLOCK_COUNT, 'block_count': GPU_BLOCK_COUNT, 'dtype': GPU_DTYPE, 'repeats': GPU_REPEATS},
        'runs': runs,
        'decision': {'decision': 'measure', 'reason': '仅观察代表性 KV block 的显存占用，不代表真实 backend 调度结论。'},
    }
    output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('GPU block probe 未启动：将 RUN_GPU_BLOCK_PROBE 改为 True 后运行。')


#### 5.3 读取结果与记录证据

读取执行单元生成的 JSON，先核对 block shape、dtype 和 evidence level，再填写下方记录表。没有实际运行结果时保留待复测状态。

In [ ]:
# 5.3：读取 GPU block probe JSON；默认没有结果时保留待复测状态。
import json
from pathlib import Path
result_path = Path(GPU_RESULT_PATH)
if result_path.exists():
    report = json.loads(result_path.read_text(encoding='utf-8'))
    required = {'task', 'evidence_level', 'runtime', 'config', 'runs', 'decision'}
    missing = required - set(report)
    if missing:
        raise ValueError(f'结果 JSON 缺少字段：{sorted(missing)}')
    print(report)
else:
    print(f'尚无 GPU block probe 结果：{result_path}')


#### GPU 实验记录

完成执行单元后，每次改变 block 数量、token 数或 dtype 都新增一行；不要覆盖已有记录。

| 配置组 | GPU / dtype | block shape | block 数量 | 理论单 block 字节数 | 峰值显存（MB） | 结果文件 |
| --- | --- | --- | ---: | ---: | ---: | --- |
| 例：本机 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 |
|  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |


## 相关阅读

完成缓存状态、价值评分和容量驱逐的模拟后，可以继续阅读 PagedAttention、前缀缓存和服务调度的实现。

- [PagedAttention 原论文：Efficient Memory Management for Large Language Model Serving](https://arxiv.org/abs/2309.06180)
- [vLLM 官方仓库](https://github.com/vllm-project/vllm)
- [38. Prefill/Decode Scheduling | Prefill/Decode 调度](./38_Prefill_Decode_Scheduling.ipynb)
- [39. Hetero PD and Serving Tiers | 异构 PD 与服务分层](./39_Hetero_PD_and_Serving_Tiers.ipynb)
- [70. Serving Scheduler Benchmark | 服务调度基准项目](./70_Serving_Scheduler_Benchmark.ipynb)
